# Comprehensive 5-Fold Cross-Validation: All Experiments

**Goal:** Validate all findings with 5-fold CV for robust, reproducible results.

**12 Experiment Configurations:**
1. EfficientNet-B0 (6-class baseline)
2. EfficientNet-B0 + Confidence Thresholding
3. EfficientNet-B0 + Confidence Thresholding + Class Imbalance Mitigation
4. EfficientNet-B0 + Confidence Thresholding + CutMix
5. EfficientNet-B0 + Confidence Thresholding + Imbalance + CutMix
6. DINOv2 Frozen (6-class)
7. DINOv2 Frozen 5-class + Threshold
8. DINOv2 Frozen 5-class + Threshold + CutMix
9. DINOv2 Unfreeze-1 5-class + Threshold
10. DINOv2 Unfreeze-1 5-class + Threshold + CutMix
11. DINOv2 Unfreeze-1 5-class + Threshold + k-NN Ensemble
12. DINOv2 Unfreeze-1 5-class + Threshold + k-NN Ensemble + CutMix

---

## 1. Setup

In [14]:
import os
import random
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 4080 Laptop GPU


## 2. Data: Merge Train + Test for Cross-Validation

In [15]:
TRAIN_DIR = './data_train'
TEST_DIR = './data_test'

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 25
PATIENCE = 5
N_FOLDS = 5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [16]:
# Collect ALL image paths and labels from both train and test
all_samples = []  # list of (path, label_idx)

# Use train dir to define class ordering
CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
class_to_idx = {cls: i for i, cls in enumerate(CLASS_NAMES)}

CLEAN_CLASS_NAMES = [c for c in CLASS_NAMES if c != 'other']
OTHER_IDX = class_to_idx['other']

# 5-class label mapping
old_to_new = {}
ni = 0
for oi, cls in enumerate(CLASS_NAMES):
    if cls != 'other':
        old_to_new[oi] = ni
        ni += 1
new_to_old = {v: k for k, v in old_to_new.items()}

for data_dir in [TRAIN_DIR, TEST_DIR]:
    for cls in CLASS_NAMES:
        cls_path = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        for fname in sorted(os.listdir(cls_path)):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                all_samples.append((os.path.join(cls_path, fname), class_to_idx[cls]))

all_paths = [s[0] for s in all_samples]
all_labels = np.array([s[1] for s in all_samples])

print(f'Total images: {len(all_samples)}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Distribution:')
for idx, cls in enumerate(CLASS_NAMES):
    cnt = (all_labels == idx).sum()
    print(f'  {cls:30s}: {cnt}')

Total images: 319
Classes (6): ['em_be_choi_verified', 'ngay_tet_verified', 'other', 'thiennhien', 'trekking_verified', 'tu_hop_verified']
Distribution:
  em_be_choi_verified           : 59
  ngay_tet_verified             : 51
  other                         : 39
  thiennhien                    : 80
  trekking_verified             : 50
  tu_hop_verified               : 40


In [17]:
# Custom dataset that loads from path list
class PathDataset(torch.utils.data.Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


# Create 5-fold splits (stratified by class)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(all_paths, all_labels))

print(f'\n{N_FOLDS}-fold split created.')
for i, (train_idx, val_idx) in enumerate(folds):
    print(f'  Fold {i+1}: train={len(train_idx)}, val={len(val_idx)}')


5-fold split created.
  Fold 1: train=255, val=64
  Fold 2: train=255, val=64
  Fold 3: train=255, val=64
  Fold 4: train=255, val=64
  Fold 5: train=256, val=63


## 3. Model Definitions

In [18]:
# Load backbones once (shared across all folds)
print('Loading DINOv2...')
dinov2_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
print('Done.')

Loading DINOv2...


Using cache found in C:\Users\Quang Dao/.cache\torch\hub\facebookresearch_dinov2_main


Done.


In [19]:
def build_efficientnet(num_classes, freeze_backbone=True):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes)
    )
    return model


class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, embed_dim=384):
        super().__init__()
        self.backbone = backbone
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone.eval()
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.classifier(features)
    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self
    def extract_features(self, x):
        with torch.no_grad():
            return self.backbone(x)


class DINOv2PartialFinetune(nn.Module):
    def __init__(self, backbone, num_classes, unfreeze_last_n=1, embed_dim=384):
        super().__init__()
        self.backbone = backbone
        self.unfreeze_last_n = unfreeze_last_n
        num_blocks = len(backbone.blocks)
        for param in self.backbone.parameters():
            param.requires_grad = False
        for i in range(num_blocks - unfreeze_last_n, num_blocks):
            for param in self.backbone.blocks[i].parameters():
                param.requires_grad = True
        for param in self.backbone.norm.parameters():
            param.requires_grad = True
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)
    def extract_features(self, x):
        return self.backbone(x)

## 4. Training & Evaluation Utilities

In [20]:
# --- CutMix ---
def rand_bbox(size, lam):
    H, W = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_h, cut_w = int(H * cut_rat), int(W * cut_rat)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    return y1, y2, x1, x2

def apply_cutmix(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(images.size(0)).to(images.device)
    y1, y2, x1, x2 = rand_bbox(images.size(), lam)
    images[:, :, y1:y2, x1:x2] = images[index, :, y1:y2, x1:x2]
    lam = 1 - ((y2 - y1) * (x2 - x1) / (images.size(-1) * images.size(-2)))
    return images, labels, labels[index], lam

In [21]:
def train_one_epoch(model, loader, criterion, optimizer, device, cutmix_prob=0.0):
    model.train()
    running_loss = 0.0; correct = 0; total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if cutmix_prob > 0 and np.random.rand() < cutmix_prob:
            images, la, lb, lam = apply_cutmix(images, labels)
            outputs = model(images)
            loss = lam * criterion(outputs, la) + (1 - lam) * criterion(outputs, lb)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += (lam * pred.eq(la).sum().item() + (1-lam) * pred.eq(lb).sum().item())
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()
    
    return running_loss / total, correct / total

In [22]:
@torch.no_grad()
def evaluate_6cls(model, loader, device):
    """Standard 6-class evaluation."""
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        outputs = model(images.to(device))
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def evaluate_with_threshold(model, loader, threshold, device):
    """5-class model → 6-class via confidence thresholding."""
    model.eval()
    all_preds, all_labels, all_confs = [], [], []
    for images, labels in loader:
        logits = model(images.to(device))
        probs = torch.softmax(logits, dim=1)
        max_p, pred_5 = probs.max(dim=1)
        for i in range(len(labels)):
            c = max_p[i].item()
            all_confs.append(c)
            all_labels.append(labels[i].item())
            all_preds.append(OTHER_IDX if c < threshold else new_to_old[pred_5[i].item()])
    return np.array(all_preds), np.array(all_labels), np.array(all_confs)


@torch.no_grad()
def evaluate_knn_ensemble(model, train_loader_eval, test_loader, threshold, 
                          clf_weight, k, device):
    """k-NN + classifier ensemble with thresholding."""
    model.eval()
    
    # Extract train features
    train_feats, train_labs = [], []
    for images, labels in train_loader_eval:
        feats = model.extract_features(images.to(device))
        train_feats.append(feats.cpu().numpy())
        train_labs.extend(labels.numpy())
    train_feats = normalize(np.vstack(train_feats), norm='l2')
    train_labs = np.array(train_labs)
    
    # Fit k-NN
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine', weights='distance')
    knn.fit(train_feats, train_labs)
    
    # Extract test features + classifier probs
    test_feats_list, clf_probs_list, all_labels = [], [], []
    for images, labels in test_loader:
        feats = model.extract_features(images.to(device))
        logits = model(images.to(device))
        probs = torch.softmax(logits, dim=1)
        test_feats_list.append(feats.cpu().numpy())
        clf_probs_list.append(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
    
    test_feats = normalize(np.vstack(test_feats_list), norm='l2')
    clf_probs = np.vstack(clf_probs_list)
    knn_probs = knn.predict_proba(test_feats)
    all_labels = np.array(all_labels)
    
    # Ensemble
    ensemble_probs = clf_weight * clf_probs + (1 - clf_weight) * knn_probs
    max_p = ensemble_probs.max(axis=1)
    pred_5 = ensemble_probs.argmax(axis=1)
    
    all_preds = []
    for i in range(len(all_labels)):
        all_preds.append(OTHER_IDX if max_p[i] < threshold else new_to_old[pred_5[i]])
    
    return np.array(all_preds), all_labels


def find_best_threshold(model, loader, device, mode='threshold'):
    """Sweep thresholds and return the best one."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.40, 0.95, 0.05):
        preds, labels, _ = evaluate_with_threshold(model, loader, t, device)
        mf1 = f1_score(labels, preds, average='macro')
        if mf1 > best_f1:
            best_f1 = mf1; best_t = t
    return best_t

## 5. Experiment Runner

In [ ]:
def run_single_fold(config, fold_idx, train_indices, val_indices):
    """
    Run a single fold for a given experiment configuration.
    Returns macro_f1, accuracy, per-class f1.
    """
    # --- Prepare fold data ---
    train_paths = [all_paths[i] for i in train_indices]
    train_labs  = all_labels[train_indices]
    val_paths   = [all_paths[i] for i in val_indices]
    val_labs    = all_labels[val_indices]
    
    use_threshold = config.get('threshold', False)
    use_imbalance = config.get('imbalance', False)
    cutmix_prob   = config.get('cutmix', 0.0)
    use_knn       = config.get('knn', False)
    
    # Filter to 5-class if using threshold
    if use_threshold:
        clean_mask = train_labs != OTHER_IDX
        tr_paths = [p for p, m in zip(train_paths, clean_mask) if m]
        tr_labs  = np.array([old_to_new[l] for l, m in zip(train_labs, clean_mask) if m])
        num_out_classes = len(CLEAN_CLASS_NAMES)
    else:
        tr_paths = train_paths
        tr_labs  = train_labs
        num_out_classes = NUM_CLASSES
    
    # Datasets
    train_ds = PathDataset(tr_paths, tr_labs, transform=train_transform)
    val_ds   = PathDataset(val_paths, val_labs, transform=eval_transform)
    
    # Class imbalance: weighted sampler + weighted loss
    if use_imbalance:
        counts = Counter(tr_labs.tolist())
        n_samples = len(tr_labs)
        n_cls = num_out_classes
        cls_weights = {c: n_samples / (n_cls * cnt) for c, cnt in counts.items()}
        sample_weights = [cls_weights[int(l)] for l in tr_labs]
        sampler = torch.utils.data.WeightedRandomSampler(sample_weights, n_samples, replacement=True)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=0, pin_memory=True)
        loss_w = torch.tensor([cls_weights[i] for i in range(n_cls)], dtype=torch.float32).to(device)
        loss_w = loss_w / loss_w.sum() * n_cls
        criterion = nn.CrossEntropyLoss(weight=loss_w)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True)
        criterion = nn.CrossEntropyLoss()
    
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=True)
    
    # --- Build model ---
    backbone_type = config['backbone']
    
    if backbone_type == 'efficientnet':
        model = build_efficientnet(num_out_classes).to(device)
        # 2-phase: freeze then unfreeze
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=1e-4, weight_decay=1e-4)
        unfreeze_epoch = 5
    elif backbone_type == 'dinov2_frozen':
        model = DINOv2Classifier(dinov2_backbone, num_out_classes).to(device)
        optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
        unfreeze_epoch = None
    elif backbone_type == 'dinov2_unfreeze1':
        model = DINOv2PartialFinetune(dinov2_backbone, num_out_classes, unfreeze_last_n=1).to(device)
        bb_params = list(model.backbone.blocks[11].parameters()) + list(model.backbone.norm.parameters())
        optimizer = optim.AdamW([
            {'params': bb_params, 'lr': 1e-5},
            {'params': model.classifier.parameters(), 'lr': 1e-3},
        ], weight_decay=1e-4)
        unfreeze_epoch = None
    
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)
    
    # --- Train ---
    best_acc = 0.0; patience_cnt = 0
    
    for epoch in range(1, NUM_EPOCHS + 1):
        # EfficientNet phase 2: unfreeze backbone
        if unfreeze_epoch and epoch == unfreeze_epoch + 1:
            for param in model.features.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW([
                {'params': model.features.parameters(), 'lr': 1e-5},
                {'params': model.classifier.parameters(), 'lr': 1e-4},
            ], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=NUM_EPOCHS - unfreeze_epoch, T_mult=1, eta_min=1e-6)
        
        _, acc = train_one_epoch(model, train_loader, criterion, optimizer, device,
                                 cutmix_prob=cutmix_prob)
        scheduler.step()
        
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        
        if patience_cnt >= PATIENCE and (unfreeze_epoch is None or epoch > unfreeze_epoch):
            break
    
    # Load best
    model.load_state_dict(best_state)
    
    # --- Evaluate ---
    if use_threshold:
        if use_knn:
            # k-NN ensemble: need eval-transform train loader
            tr_eval_ds = PathDataset(tr_paths, tr_labs, transform=eval_transform)
            tr_eval_loader = DataLoader(tr_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                        num_workers=0, pin_memory=True)
            # Sweep threshold with k-NN
            best_t, best_f1 = 0.5, 0.0
            for t in np.arange(0.40, 0.95, 0.05):
                preds, labs = evaluate_knn_ensemble(
                    model, tr_eval_loader, val_loader, t, 0.9, 5, device)
                mf1 = f1_score(labs, preds, average='macro')
                if mf1 > best_f1:
                    best_f1 = mf1; best_t = t
            preds, labs = evaluate_knn_ensemble(
                model, tr_eval_loader, val_loader, best_t, 0.9, 5, device)
        else:
            best_t = find_best_threshold(model, val_loader, device)
            preds, labs, _ = evaluate_with_threshold(model, val_loader, best_t, device)
    else:
        preds, labs = evaluate_6cls(model, val_loader, device)
    
    macro_f1 = f1_score(labs, preds, average='macro')
    accuracy = (preds == labs).mean()
    per_class_f1 = f1_score(labs, preds, average=None, labels=list(range(NUM_CLASSES)))
    
    return macro_f1, accuracy, per_class_f1

In [24]:
def run_experiment_cv(config, folds):
    """Run all folds for an experiment configuration."""
    name = config['name']
    print(f'\n{"="*70}')
    print(f'  {name}')
    print(f'{"="*70}')
    
    fold_f1s, fold_accs, fold_pcf1s = [], [], []
    
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        t0 = time.time()
        mf1, acc, pcf1 = run_single_fold(config, fold_idx, train_idx, val_idx)
        elapsed = time.time() - t0
        
        fold_f1s.append(mf1)
        fold_accs.append(acc)
        fold_pcf1s.append(pcf1)
        
        print(f'  Fold {fold_idx+1}: Macro F1={mf1:.4f}, Acc={acc:.4f} ({elapsed:.0f}s)')
    
    mean_f1 = np.mean(fold_f1s)
    std_f1  = np.std(fold_f1s)
    mean_acc = np.mean(fold_accs)
    std_acc  = np.std(fold_accs)
    mean_pcf1 = np.mean(fold_pcf1s, axis=0)
    
    print(f'\n  MEAN: Macro F1={mean_f1:.4f} +/- {std_f1:.4f}, Acc={mean_acc:.4f} +/- {std_acc:.4f}')
    
    return {
        'name': name,
        'fold_f1s': fold_f1s,
        'fold_accs': fold_accs,
        'mean_f1': mean_f1,
        'std_f1': std_f1,
        'mean_acc': mean_acc,
        'std_acc': std_acc,
        'mean_per_class_f1': mean_pcf1,
    }

## 6. Define All 12 Experiments

In [25]:
EXPERIMENTS = [
    # --- EfficientNet Variants ---
    {
        'name': '1. EfficientNet-B0 (6-class baseline)',
        'backbone': 'efficientnet',
        'threshold': False, 'imbalance': False, 'cutmix': 0.0, 'knn': False,
    },
    {
        'name': '2. EfficientNet-B0 + Threshold',
        'backbone': 'efficientnet',
        'threshold': True, 'imbalance': False, 'cutmix': 0.0, 'knn': False,
    },
    {
        'name': '3. EfficientNet-B0 + Threshold + Imbalance',
        'backbone': 'efficientnet',
        'threshold': True, 'imbalance': True, 'cutmix': 0.0, 'knn': False,
    },
    {
        'name': '4. EfficientNet-B0 + Threshold + CutMix',
        'backbone': 'efficientnet',
        'threshold': True, 'imbalance': False, 'cutmix': 0.7, 'knn': False,
    },
    {
        'name': '5. EfficientNet-B0 + Threshold + Imbalance + CutMix',
        'backbone': 'efficientnet',
        'threshold': True, 'imbalance': True, 'cutmix': 0.7, 'knn': False,
    },
    # --- DINOv2 Frozen Variants ---
    {
        'name': '6. DINOv2 Frozen (6-class)',
        'backbone': 'dinov2_frozen',
        'threshold': False, 'imbalance': False, 'cutmix': 0.0, 'knn': False,
    },
    {
        'name': '7. DINOv2 Frozen + Threshold',
        'backbone': 'dinov2_frozen',
        'threshold': True, 'imbalance': False, 'cutmix': 0.0, 'knn': False,
    },
    {
        'name': '8. DINOv2 Frozen + Threshold + CutMix',
        'backbone': 'dinov2_frozen',
        'threshold': True, 'imbalance': False, 'cutmix': 0.7, 'knn': False,
    },
    # --- DINOv2 Unfreeze-1 Variants ---
    {
        'name': '9. DINOv2 Unfreeze-1 + Threshold',
        'backbone': 'dinov2_unfreeze1',
        'threshold': True, 'imbalance': False, 'cutmix': 0.0, 'knn': False,
    },
    {
        'name': '10. DINOv2 Unfreeze-1 + Threshold + CutMix',
        'backbone': 'dinov2_unfreeze1',
        'threshold': True, 'imbalance': False, 'cutmix': 0.7, 'knn': False,
    },
    {
        'name': '11. DINOv2 Unfreeze-1 + Threshold + k-NN',
        'backbone': 'dinov2_unfreeze1',
        'threshold': True, 'imbalance': False, 'cutmix': 0.0, 'knn': True,
    },
    {
        'name': '12. DINOv2 Unfreeze-1 + Threshold + k-NN + CutMix',
        'backbone': 'dinov2_unfreeze1',
        'threshold': True, 'imbalance': False, 'cutmix': 0.7, 'knn': True,
    },
]

print(f'{len(EXPERIMENTS)} experiments x {N_FOLDS} folds = {len(EXPERIMENTS) * N_FOLDS} training runs')

12 experiments x 5 folds = 60 training runs


## 7. Run All Experiments

In [26]:
all_results = []
total_start = time.time()

for config in EXPERIMENTS:
    result = run_experiment_cv(config, folds)
    all_results.append(result)
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

total_time = time.time() - total_start
print(f'\n\nAll experiments complete. Total time: {total_time/60:.1f} minutes')


  1. EfficientNet-B0 (6-class baseline)


TypeError: CosineAnnealingWarmRestarts.__init__() got an unexpected keyword argument 'T_max'

## 8. Results: Full Comparison Table

In [ ]:
summary_df = pd.DataFrame([{
    'Experiment': r['name'],
    'Macro F1 (mean)': r['mean_f1'],
    'Macro F1 (std)': r['std_f1'],
    'Macro F1': f"{r['mean_f1']:.4f} ± {r['std_f1']:.4f}",
    'Accuracy': f"{r['mean_acc']:.4f} ± {r['std_acc']:.4f}",
} for r in all_results]).sort_values('Macro F1 (mean)', ascending=False)

print('=' * 95)
print('5-FOLD CROSS-VALIDATION RESULTS (sorted by Macro F1)')
print('=' * 95)
print(summary_df[['Experiment', 'Macro F1', 'Accuracy']].to_string(index=False))
print('=' * 95)

In [ ]:
# Bar chart with error bars
fig, ax = plt.subplots(figsize=(14, 8))

# Sort by mean F1
sorted_results = sorted(all_results, key=lambda r: r['mean_f1'])
names = [r['name'] for r in sorted_results]
means = [r['mean_f1'] for r in sorted_results]
stds  = [r['std_f1'] for r in sorted_results]

y = np.arange(len(names))
bars = ax.barh(y, means, xerr=stds, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(names))),
               edgecolor='black', linewidth=0.5, capsize=4)

ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel('Macro F1 (mean ± std)', fontsize=11)
ax.set_title('5-Fold CV: All Experiments Ranked by Macro F1', fontweight='bold', fontsize=13)
ax.set_xlim(0.5, 1.0)
ax.grid(axis='x', alpha=0.3)

for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(m + s + 0.01, i, f'{m:.4f}±{s:.4f}', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('cv_all_experiments.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Per-Class F1 Analysis

In [ ]:
# Per-class F1 heatmap
pcf1_data = {}
for r in all_results:
    pcf1_data[r['name']] = r['mean_per_class_f1']

pcf1_df = pd.DataFrame(pcf1_data, index=CLASS_NAMES).T

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pcf1_df, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
            cbar_kws={'label': 'Mean F1 across 5 folds'},
            linewidths=0.5, linecolor='white')
ax.set_title('Per-Class F1 Score by Experiment (5-Fold CV Mean)', fontweight='bold', fontsize=12)
ax.set_xlabel('Class')
ax.set_ylabel('Experiment')
plt.tight_layout()
plt.savefig('cv_per_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Fold Stability Analysis

In [ ]:
# Box plot of fold-level F1 scores
fig, ax = plt.subplots(figsize=(14, 7))

fold_data = [r['fold_f1s'] for r in all_results]
exp_names = [r['name'].split('. ')[1] if '. ' in r['name'] else r['name'] for r in all_results]

bp = ax.boxplot(fold_data, vert=True, patch_artist=True, labels=exp_names,
                boxprops=dict(facecolor='lightblue', color='navy'),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='navy'),
                capprops=dict(color='navy'))

# Overlay individual fold points
for i, folds_data in enumerate(fold_data):
    x = np.random.normal(i + 1, 0.04, size=len(folds_data))
    ax.scatter(x, folds_data, alpha=0.6, s=30, color='darkblue', zorder=5)

ax.set_ylabel('Macro F1')
ax.set_title('Fold-Level Macro F1 Distribution', fontweight='bold', fontsize=13)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('cv_fold_stability.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Statistical Significance

In [ ]:
from scipy import stats

# Paired t-test: best experiment vs each other
best_exp = max(all_results, key=lambda r: r['mean_f1'])
print(f'Best experiment: {best_exp["name"]}')
print(f'Mean Macro F1: {best_exp["mean_f1"]:.4f} ± {best_exp["std_f1"]:.4f}')
print(f'\nPaired t-tests (best vs each):\n')
print(f'{"Experiment":>55} | {"Mean F1":>8} | {"p-value":>8} | {"Significant?":>12}')
print('-' * 95)

for r in sorted(all_results, key=lambda x: x['mean_f1'], reverse=True):
    if r['name'] == best_exp['name']:
        continue
    t_stat, p_val = stats.ttest_rel(best_exp['fold_f1s'], r['fold_f1s'])
    sig = 'YES (p<0.05)' if p_val < 0.05 else 'no'
    print(f'{r["name"]:>55} | {r["mean_f1"]:>8.4f} | {p_val:>8.4f} | {sig:>12}')

## 12. Summary

### What this notebook validates:
- All 12 experiment configurations tested with 5-fold stratified CV
- Results reported as mean ± std across folds
- Per-class F1 heatmap shows which classes each method handles best
- Box plots show fold-level variance (stability)
- Paired t-tests check if top results are statistically significant

### Reading the results:
- **Higher mean F1** = better overall performance
- **Lower std** = more stable/reliable across different data splits
- **p < 0.05** in paired t-test = the difference is statistically significant, not just noise
- The per-class heatmap reveals which improvements target which classes